In [1]:
import pandas as pd

running = pd.read_csv("../data/processed/bigwest_2021_2026_running.csv")
field = pd.read_csv("../data/processed/bigwest_2021_2026_field.csv")

athletes = pd.concat([
    running[["athlete_id", "athlete_name"]],
    field[["athlete_id", "athlete_name"]]
]).drop_duplicates(subset=["athlete_id"])

Unique athletes: 1514


In [4]:
athlete_ids = (
    athletes["athlete_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda x: x.ne("")]
    .unique()
)

print(f"Found {len(athlete_ids)} unique athletes")

Found 1514 unique athletes


In [7]:
import requests
import os


url = "https://api.parse.bot/scraper/1b66c7cb-769a-4301-82ff-8d308afee7ad/get_athlete_results"

headers = {
    "X-API-Key": "pmx_127bea8970ed7f3b4ddaec353e4310e8",
    "API-Snapshot-Version": "5"
}

response = requests.get(
    url,
    headers=headers,
    params={"athlete_id": athlete_ids[0]},
)

response.raise_for_status()

data = response.json()
print(data)

{'status': 'success', 'data': {'athlete_id': '7657050', 'name': 'Benjamin Goodwin', 'team': 'UC DAVIS', 'year': '', 'personal_bests': [{'event': '100', 'mark': '10.67(1.3)'}, {'event': '200', 'mark': '21.87(0.4)'}], 'performances': [{'meet_name': '62nd Annual Mt. SAC Relays (Collegiate)', 'meet_date': 'Apr 13-16, 2022', 'event': '100', 'mark': '10.74(1.9)', 'mark_seconds': 10.74, 'place': '58th\n          (P)', 'is_xc': False, 'is_track_field': True}, {'meet_name': '62nd Annual Mt. SAC Relays (Collegiate)', 'meet_date': 'Apr 13-16, 2022', 'event': '4x100', 'mark': '41.06', 'mark_seconds': 41.06, 'place': '4th\n          (F)', 'is_xc': False, 'is_track_field': True}, {'meet_name': 'Brutus Hamilton Invitational', 'meet_date': 'Apr  9, 2022', 'event': '100', 'mark': '10.95(-1.1)', 'mark_seconds': 10.95, 'place': '10th\n          (F)', 'is_xc': False, 'is_track_field': True}, {'meet_name': 'Rafer Johnson & Jackie Joyner- Kersee Invite', 'meet_date': 'Mar 26, 2022', 'event': '100', 'mark': 

In [9]:
import pandas as pd

data = response.json()["data"]

athlete_df = pd.DataFrame(data["performances"])

athlete_df["athlete_id"] = data["athlete_id"]
athlete_df["athlete_name"] = data["name"]
athlete_df["team"] = data["team"]

# Remove incorrectly parsed rows
athlete_df = athlete_df[
    athlete_df["meet_date"].str.contains(
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\b",
        na=False
    )
    & athlete_df["mark_seconds"].notna()
].copy()

athlete_df = athlete_df.drop_duplicates()

athlete_df

,meet_name,meet_date,event,mark,mark_seconds,place,is_xc,is_track_field,athlete_id,athlete_name,team
0,62nd Annual Mt. SAC Relays (Collegiate),"Apr 13-16, 2022",100,10.74(1.9),10.74,58th\n (P),False,True,7657050,Benjamin Goodwin,UC DAVIS
1,62nd Annual Mt. SAC Relays (Collegiate),"Apr 13-16, 2022",4x100,41.06,41.06,4th\n (F),False,True,7657050,Benjamin Goodwin,UC DAVIS
2,Brutus Hamilton Invitational,"Apr 9, 2022",100,10.95(-1.1),10.95,10th\n (F),False,True,7657050,Benjamin Goodwin,UC DAVIS
3,Rafer Johnson & Jackie Joyner- Kersee Invite,"Mar 26, 2022",100,10.88(-0.4),10.88,12th\n (F),False,True,7657050,Benjamin Goodwin,UC DAVIS
4,Rafer Johnson & Jackie Joyner- Kersee Invite,"Mar 26, 2022",4x100,40.51,40.51,1st\n (F),False,True,7657050,Benjamin Goodwin,UC DAVIS
5,2022 Hornet Invite,"Mar 19, 2022",4x100,41.41,41.41,1st\n (F),False,True,7657050,Benjamin Goodwin,UC DAVIS
6,Aggie Open,"Mar 11-12, 2022",100,10.75(2.2),10.75,1st\n (F),False,True,7657050,Benjamin Goodwin,UC DAVIS
7,Aggie Open,"Mar 11-12, 2022",4x100,41.74,41.74,2nd\n (F),False,True,7657050,Benjamin Goodwin,UC DAVIS
8,2021 Big West Outdoor Track & Field Championships,"May 12-15, 2021",100,10.67(1.2),10.67,1st\n (F),False,True,7657050,Benjamin Goodwin,UC DAVIS
9,2021 Big West Outdoor Track & Field Championships,"May 12-15, 2021",100,10.78(-0.4),10.78,2nd\n (P),False,True,7657050,Benjamin Goodwin,UC DAVIS


In [10]:
import requests
import pandas as pd
import time

all_performances = []
failed_ids = []

for i, athlete_id in enumerate(athlete_ids):

    try:
        response = requests.get(
            url,
            headers=headers,
            params={"athlete_id": str(athlete_id)},
            timeout=120
        )

        response.raise_for_status()
        result = response.json()

        if result.get("status") != "success":
            failed_ids.append(athlete_id)
            continue

        athlete = result["data"]

        for performance in athlete.get("performances", []):

            meet_date = str(performance.get("meet_date", ""))

            # Keep only records with actual meet dates
            if not any(month in meet_date for month in [
                "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
            ]):
                continue

            all_performances.append({
                "athlete_id": athlete_id,
                "athlete_name": athlete.get("name"),
                "team": athlete.get("team"),
                **performance
            })

        print(f"Completed {i + 1}/{len(athlete_ids)}")

    except (requests.RequestException, ValueError, KeyError, TypeError) as e:
        print(f"Failed {athlete_id}: {e}")
        failed_ids.append(athlete_id)

    time.sleep(9)  # Adjust according to permitted API limits

results_df = pd.DataFrame(all_performances)

results_df = results_df.drop_duplicates()

results_df.to_csv(
    "bigwest_regular_season_results.csv",
    index=False
)

print("Total performances:", len(results_df))
print("Unique athletes:", results_df["athlete_id"].nunique())
print("Failed athletes:", len(failed_ids))

Completed 1/1514
Completed 2/1514
Completed 3/1514
Completed 4/1514
Completed 5/1514
Completed 6/1514
Completed 7/1514
Completed 8/1514
Completed 9/1514
Completed 10/1514
Completed 11/1514
Completed 12/1514
Completed 13/1514
Completed 14/1514
Completed 15/1514
Completed 16/1514
Completed 17/1514
Completed 18/1514
Completed 19/1514
Completed 20/1514
Completed 21/1514
Completed 22/1514
Completed 23/1514
Completed 24/1514
Completed 25/1514
Completed 26/1514
Completed 27/1514
Completed 28/1514
Completed 29/1514
Completed 30/1514
Completed 31/1514
Completed 32/1514
Completed 33/1514
Completed 34/1514
Completed 35/1514
Completed 36/1514
Completed 37/1514
Completed 38/1514
Completed 39/1514
Completed 40/1514
Completed 41/1514
Failed 6886461: 429 Client Error: Too Many Requests for url: https://api.parse.bot/scraper/1b66c7cb-769a-4301-82ff-8d308afee7ad/get_athlete_results?athlete_id=6886461
Failed 5471935: 429 Client Error: Too Many Requests for url: https://api.parse.bot/scraper/1b66c7cb-769a

KeyboardInterrupt: 

In [14]:
results_df = pd.DataFrame(all_performances)

results_df.to_csv(
    "bigwest_partial_results.csv",
    index=False
)

pd.DataFrame({
    "athlete_id": failed_ids
}).to_csv(
    "bigwest_failed_ids.csv",
    index=False
)
results_df

,athlete_id,athlete_name,team,meet_name,meet_date,event,mark,mark_seconds,place,is_xc,is_track_field
0,7657050,Benjamin Goodwin,UC DAVIS,62nd Annual Mt. SAC Relays (Collegiate),"Apr 13-16, 2022",100,10.74(1.9),10.74,58th\n (P),False,True
1,7657050,Benjamin Goodwin,UC DAVIS,62nd Annual Mt. SAC Relays (Collegiate),"Apr 13-16, 2022",4x100,41.06,41.06,4th\n (F),False,True
2,7657050,Benjamin Goodwin,UC DAVIS,Brutus Hamilton Invitational,"Apr 9, 2022",100,10.95(-1.1),10.95,10th\n (F),False,True
3,7657050,Benjamin Goodwin,UC DAVIS,Rafer Johnson & Jackie Joyner- Kersee Invite,"Mar 26, 2022",100,10.88(-0.4),10.88,12th\n (F),False,True
4,7657050,Benjamin Goodwin,UC DAVIS,Rafer Johnson & Jackie Joyner- Kersee Invite,"Mar 26, 2022",4x100,40.51,40.51,1st\n (F),False,True
...,...,...,...,...,...,...,...,...,...,...,...
4330,7907793,Allie Scimia,LONG BEACH ST.,Big West Cross Country Championships,"Oct 29, 2021",6k,22:09.6,1329.60,32nd,True,False
4331,7907793,Allie Scimia,LONG BEACH ST.,2021 Highlander Invitational,"Oct 16, 2021",6k,21:20.1,1280.10,20th,True,False
4332,7907793,Allie Scimia,LONG BEACH ST.,Chile Pepper XC Festival,"Oct 1, 2021",5k,17:28.9,1048.90,22nd,True,False
4333,7907793,Allie Scimia,LONG BEACH ST.,UCR 40th Annual Cross Country Invitational,"Sep 18, 2021",6k,21:18.0,1278.00,29th,True,False


In [15]:
print("Status:", response.status_code)
print("Retry-After:", response.headers.get("Retry-After"))
print("Rate-limit headers:")

for key, value in response.headers.items():
    if "limit" in key.lower() or "retry" in key.lower():
        print(key, ":", value)

print("Response:", response.text[:1000])

Status: 429
Retry-After: 9
Rate-limit headers:
retry-after : 9
x-ratelimit-burst : 30
x-ratelimit-daily-limit : 100
x-ratelimit-daily-remaining : 21
x-ratelimit-daily-reset : 24575
x-ratelimit-limit : 30
x-ratelimit-refill-per-min : 5
x-ratelimit-remaining : 0
x-ratelimit-reset : 1792516234
Response: {"error":{"error":"Rate limit exceeded","message":"Too many requests in a short burst (bucket of 30, refilling at 5/min). Retry in 9s.","limit_type":"burst","retry_after":9,"upgrade_hint":"You're at 5 requests/min. Upgrade to Hobby for 20 requests/min and 1,000 credits/month."},"status_code":429}
